In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [11]:
df = pd.read_csv('1stRoundFlightdata.csv')
X = df.drop(columns=["price"])
y = df["price"]
print(X.shape, y.shape)

(14431, 65) (14431,)


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=X["flight_class"],
)
print(X_train.shape, X_test.shape)

(11544, 65) (2887, 65)


In [13]:
def evaluate(y_true, y_pred, name="model"):
    # log-scale metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    # back to rupees
    true_rs, pred_rs = np.expm1(y_true), np.expm1(y_pred)
    mae_rs = mean_absolute_error(true_rs, pred_rs)
    mape = np.mean(np.abs(true_rs - pred_rs) / true_rs) * 100
    print(f"{name}: log RMSE={rmse:.4f} | log MAE={mae:.4f} | R2={r2:.4f} | "
          f"MAE=Rs {mae_rs:,.0f} | MAPE={mape:.1f}%")

# Baseline: mean log price per class, learned from the training set only
class_means = y_train.groupby(X_train["flight_class"]).mean()
baseline_pred = X_test["flight_class"].map(class_means)

evaluate(y_test, baseline_pred, "Baseline (class mean)")

Baseline (class mean): log RMSE=0.4921 | log MAE=0.3603 | R2=0.7642 | MAE=Rs 15,380 | MAPE=34.7%


In [14]:
"""
Setup for tuning
- cv: the same 5-fold split used everywhere before (shuffled, seed 42), so scores
  stay comparable with the earlier tables.
- cv_metrics: trains a fresh copy of a model on each fold and returns the five
  metrics (log RMSE, log MAE, R2, MAE in rupees, MAPE) for every fold.
"""
from sklearn.base import clone
from sklearn.model_selection import KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_metrics(model, X, y, cv):
    rows = []
    for tr, va in cv.split(X):
        m = clone(model).fit(X.iloc[tr], y.iloc[tr])
        pred = m.predict(X.iloc[va])
        true = y.iloc[va]
        true_rs, pred_rs = np.expm1(true), np.expm1(pred)
        rows.append({
            "log RMSE": np.sqrt(mean_squared_error(true, pred)),
            "log MAE": mean_absolute_error(true, pred),
            "R2": r2_score(true, pred),
            "MAE (Rs)": mean_absolute_error(true_rs, pred_rs),
            "MAPE (%)": np.mean(np.abs(true_rs - pred_rs) / true_rs) * 100,
        })
    return pd.DataFrame(rows)

In [15]:
"""
Tune XGBoost (1st round data)
- Randomized search tries 30 random combinations of settings.
- Each combination is scored with the same 5-fold CV (cv) on the TRAINING set
  only, so the scores are comparable with the comparison table.
- Settings searched: learning rate, number of trees, tree depth, minimum weight
  per leaf, row sampling, column sampling, L2 penalty, and gamma (minimum gain
  needed to split, which stops pointless splits).
- Prints the best settings and the 5 best combinations, then scores the best
  model with all five metrics (same as the comparison table).
- The best model is saved as best_xgb. The test set is not touched.
- To beat: untuned XGBoost log RMSE 0.1364 | MAE Rs 3,444 | MAPE 8.2%
"""
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

param_dist = {
    "learning_rate": [0.02, 0.03, 0.05, 0.08],
    "n_estimators": [400, 800, 1200],
    "max_depth": [4, 6, 8, 10],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.5, 0.7, 0.9],
    "reg_lambda": [0.5, 1, 3, 10],
    "gamma": [0, 0.05, 0.2],
}

search_xgb = RandomizedSearchCV(
    XGBRegressor(n_jobs=-1, random_state=42),
    param_distributions=param_dist,
    n_iter=30,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=1,
    random_state=42,
    verbose=1,
)
search_xgb.fit(X_train, y_train)

print("Best settings:", search_xgb.best_params_)
print(f"Best CV RMSE: {-search_xgb.best_score_:.4f}   (untuned XGBoost: 0.1364)")
best_xgb = search_xgb.best_estimator_

# Top 5 combinations
res = pd.DataFrame(search_xgb.cv_results_)
cols = ["mean_test_score", "std_test_score"] + [f"param_{k}" for k in param_dist]
top = res.sort_values("mean_test_score", ascending=False).head(5)[cols].copy()
top["mean_test_score"] = -top["mean_test_score"]
top = top.rename(columns={"mean_test_score": "cv_rmse", "std_test_score": "std"})
print("\nTop 5 combinations:")
print(top.round(4).to_string(index=False))

# All five metrics for the best model, same format as the comparison table
f = cv_metrics(best_xgb, X_train, y_train, cv)
mean, std = f.mean(), f.std()
print(f"\nTuned XGBoost: log RMSE={mean['log RMSE']:.4f} (+/- {std['log RMSE']:.4f}) | "
      f"log MAE={mean['log MAE']:.4f} | R2={mean['R2']:.4f} | "
      f"MAE=Rs {mean['MAE (Rs)']:,.0f} | MAPE={mean['MAPE (%)']:.1f}% (+/- {std['MAPE (%)']:.1f})")
print("Untuned XGBoost: log RMSE=0.1364 (+/- 0.0043) | log MAE=0.0829 | R2=0.9814 | "
      "MAE=Rs 3,444 | MAPE=8.2% (+/- 0.3)")

Fitting 5 folds for each of 30 candidates, totalling 150 fits


KeyboardInterrupt: 

### XGBoost tuning (1st round data)
- Randomized search, 30 combinations × 5-fold CV on the training set.
- Best tuned model: log RMSE 0.1374 (MAPE 8.4%) vs untuned 0.1364 (MAPE 8.2%).
- The difference is inside the fold-to-fold spread (~0.005), so tuning gave no real gain. The top 5 combinations are very different but score within 0.006 of each other.
- **Decision:** keep the untuned XGBoost (faster, same accuracy).

In [ ]:
"""
Tune Extra Trees (1st round data)
- Randomized search tries 20 random combinations of settings.
- Each combination is scored with the same 5-fold CV (cv) on the TRAINING set
  only, so the scores are comparable with the earlier tables.
- Settings searched: number of trees, minimum rows per leaf, max_features (the
  share of columns each split can look at), tree depth, and bootstrap (whether
  each tree trains on a random resample of the rows).
- Prints the best settings and the 5 best combinations, then scores the best
  model with all five metrics (same as the comparison table).
- The best model is saved as best_et. The test set is not touched.
- To beat: untuned Extra Trees log RMSE 0.1417 | MAE Rs 3,410 | MAPE 8.1%
"""
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import ExtraTreesRegressor

param_dist = {
    "n_estimators": [300, 500, 800],
    "min_samples_leaf": [1, 2, 3, 5],
    "max_features": [0.3, 0.5, 0.7, 1.0],
    "max_depth": [None, 20, 30],
    "bootstrap": [False, True],
}

search_et = RandomizedSearchCV(
    ExtraTreesRegressor(n_jobs=-1, random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=1,
    random_state=42,
    verbose=1,
)
search_et.fit(X_train, y_train)

print("Best settings:", search_et.best_params_)
print(f"Best CV RMSE: {-search_et.best_score_:.4f}   (untuned Extra Trees: 0.1417)")
best_et = search_et.best_estimator_

# Top 5 combinations
res = pd.DataFrame(search_et.cv_results_)
cols = ["mean_test_score", "std_test_score"] + [f"param_{k}" for k in param_dist]
top = res.sort_values("mean_test_score", ascending=False).head(5)[cols].copy()
top["mean_test_score"] = -top["mean_test_score"]
top = top.rename(columns={"mean_test_score": "cv_rmse", "std_test_score": "std"})
print("\nTop 5 combinations:")
print(top.round(4).to_string(index=False))

# All five metrics for the best model, same format as the comparison table
f = cv_metrics(best_et, X_train, y_train, cv)
mean, std = f.mean(), f.std()
print(f"\nTuned Extra Trees: log RMSE={mean['log RMSE']:.4f} (+/- {std['log RMSE']:.4f}) | "
      f"log MAE={mean['log MAE']:.4f} | R2={mean['R2']:.4f} | "
      f"MAE=Rs {mean['MAE (Rs)']:,.0f} | MAPE={mean['MAPE (%)']:.1f}% (+/- {std['MAPE (%)']:.1f})")
print("Untuned Extra Trees: log RMSE=0.1417 (+/- 0.0066) | log MAE=0.0819 | R2=0.9799 | "
      "MAE=Rs 3,410 | MAPE=8.1% (+/- 0.3)")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best settings: {'n_estimators': 300, 'min_samples_leaf': 3, 'max_features': 1.0, 'max_depth': None, 'bootstrap': False}
Best CV RMSE: 0.1414   (untuned Extra Trees: 0.1417)

Top 5 combinations:
 cv_rmse    std  param_n_estimators  param_min_samples_leaf  param_max_features param_max_depth  param_bootstrap
  0.1414 0.0052                 300                       3                 1.0            None            False
  0.1414 0.0051                 300                       2                 1.0              30             True
  0.1416 0.0055                 500                       1                 1.0            None             True
  0.1442 0.0047                 500                       2                 0.7            None             True
  0.1465 0.0045                 300                       5                 1.0            None            False

Tuned Extra Trees: log RMSE=0.1414 (+/- 0.0058) | log MAE=0.0835 

Once again Tuning does does improve upon the existing model.

In [16]:
from sklearn.metrics import r2_score, make_scorer
import numpy as np

def raw_price_r2(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return r2_score(y_true, y_pred)

kaggle_scorer = make_scorer(raw_price_r2)

In [17]:
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

param_dist = {
    "learning_rate": [0.02, 0.03, 0.05, 0.08],
    "n_estimators": [400, 800, 1200],
    "max_depth": [4, 6, 8, 10],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.5, 0.7, 0.9],
    "reg_lambda": [0.5, 1, 3, 10],
    "gamma": [0, 0.05, 0.2],
}

search_xgb = RandomizedSearchCV(
    XGBRegressor(n_jobs=-1, random_state=42),
    param_distributions=param_dist,
    n_iter=30,
    cv=cv,
    scoring=kaggle_scorer,
    n_jobs=1,
    random_state=42,
    verbose=1,
)
search_xgb.fit(X_train, y_train)

print("Best settings:", search_xgb.best_params_)
print(f"Best CV RMSE: {-search_xgb.best_score_:.4f}   (untuned XGBoost: 0.1364)")
best_xgb = search_xgb.best_estimator_

# Top 5 combinations
res = pd.DataFrame(search_xgb.cv_results_)
cols = ["mean_test_score", "std_test_score"] + [f"param_{k}" for k in param_dist]
top = res.sort_values("mean_test_score", ascending=False).head(5)[cols].copy()
top["mean_test_score"] = -top["mean_test_score"]
top = top.rename(columns={"mean_test_score": "cv_rmse", "std_test_score": "std"})
print("\nTop 5 combinations:")
print(top.round(4).to_string(index=False))

# All five metrics for the best model, same format as the comparison table
f = cv_metrics(best_xgb, X_train, y_train, cv)
mean, std = f.mean(), f.std()
print(f"\nTuned XGBoost: log RMSE={mean['log RMSE']:.4f} (+/- {std['log RMSE']:.4f}) | "
      f"log MAE={mean['log MAE']:.4f} | R2={mean['R2']:.4f} | "
      f"MAE=Rs {mean['MAE (Rs)']:,.0f} | MAPE={mean['MAPE (%)']:.1f}% (+/- {std['MAPE (%)']:.1f})")
print("Untuned XGBoost: log RMSE=0.1364 (+/- 0.0043) | log MAE=0.0829 | R2=0.9814 | "
      "MAE=Rs 3,444 | MAPE=8.2% (+/- 0.3)")

import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import ExtraTreesRegressor

param_dist = {
    "n_estimators": [300, 500, 800],
    "min_samples_leaf": [1, 2, 3, 5],
    "max_features": [0.3, 0.5, 0.7, 1.0],
    "max_depth": [None, 20, 30],
    "bootstrap": [False, True],
}

search_et = RandomizedSearchCV(
    ExtraTreesRegressor(n_jobs=-1, random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=cv,
    scoring=kaggle_scorer,
    n_jobs=1,
    random_state=42,
    verbose=1,
)
search_et.fit(X_train, y_train)

print("Best settings:", search_et.best_params_)
print(f"Best CV RMSE: {-search_et.best_score_:.4f}   (untuned Extra Trees: 0.1417)")
best_et = search_et.best_estimator_

# Top 5 combinations
res = pd.DataFrame(search_et.cv_results_)
cols = ["mean_test_score", "std_test_score"] + [f"param_{k}" for k in param_dist]
top = res.sort_values("mean_test_score", ascending=False).head(5)[cols].copy()
top["mean_test_score"] = -top["mean_test_score"]
top = top.rename(columns={"mean_test_score": "cv_rmse", "std_test_score": "std"})
print("\nTop 5 combinations:")
print(top.round(4).to_string(index=False))

# All five metrics for the best model, same format as the comparison table
f = cv_metrics(best_et, X_train, y_train, cv)
mean, std = f.mean(), f.std()
print(f"\nTuned Extra Trees: log RMSE={mean['log RMSE']:.4f} (+/- {std['log RMSE']:.4f}) | "
      f"log MAE={mean['log MAE']:.4f} | R2={mean['R2']:.4f} | "
      f"MAE=Rs {mean['MAE (Rs)']:,.0f} | MAPE={mean['MAPE (%)']:.1f}% (+/- {std['MAPE (%)']:.1f})")
print("Untuned Extra Trees: log RMSE=0.1417 (+/- 0.0066) | log MAE=0.0819 | R2=0.9799 | "
      "MAE=Rs 3,410 | MAPE=8.1% (+/- 0.3)")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best settings: {'subsample': 0.8, 'reg_lambda': 0.5, 'n_estimators': 800, 'min_child_weight': 3, 'max_depth': 6, 'learning_rate': 0.03, 'gamma': 0, 'colsample_bytree': 0.5}
Best CV RMSE: -0.9325   (untuned XGBoost: 0.1364)

Top 5 combinations:
 cv_rmse    std  param_learning_rate  param_n_estimators  param_max_depth  param_min_child_weight  param_subsample  param_colsample_bytree  param_reg_lambda  param_gamma
 -0.9325 0.0058                 0.03                 800                6                       3              0.8                     0.5               0.5         0.00
 -0.9315 0.0104                 0.05                1200               10                       5              1.0                     0.5               1.0         0.00
 -0.9314 0.0071                 0.05                 400                8                       1              0.6                     0.9               0.5         0.05
 -0.9307 0.007

In [18]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    make_scorer
)
from sklearn.ensemble import ExtraTreesRegressor
from xgboost import XGBRegressor

from scipy.stats import randint, uniform

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv("1stRoundFlightdata.csv")

TARGET = "price"

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(float)

print("Dataset shape:", df.shape)
print("Feature shape:", X.shape)
print("Target range:", y.min(), "to", y.max())

# Your uploaded dataset appears to contain log1p(price).
# Do NOT apply np.log1p() again.
# This code assumes y = log1p(original_price).

# Both tree models require numeric input.
non_numeric = X.select_dtypes(exclude=np.number).columns.tolist()

if non_numeric:
    raise ValueError(
        f"Encode these non-numeric columns first: {non_numeric}"
    )

if not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError("Features contain NaN or infinite values.")

if not np.isfinite(y.to_numpy()).all():
    raise ValueError("Target contains NaN or infinite values.")

# ============================================================
# 2. TRAIN / HELD-OUT TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=X["flight_class"] if "flight_class" in X.columns else None
)

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)

# ============================================================
# 3. CROSS-VALIDATION
# ============================================================

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# ============================================================
# 4. SCORER: R2 ON ORIGINAL PRICE SCALE
# ============================================================

def raw_price_r2(y_true_log, y_pred_log):
    y_true_price = np.expm1(y_true_log)
    y_pred_price = np.expm1(y_pred_log)

    return r2_score(y_true_price, y_pred_price)


kaggle_scorer = make_scorer(
    raw_price_r2,
    greater_is_better=True
)

# ============================================================
# 5. EVALUATION FUNCTION
# ============================================================

def evaluate(y_true_log, y_pred_log, model_name):
    y_true_price = np.expm1(np.asarray(y_true_log))
    y_pred_price = np.expm1(np.asarray(y_pred_log))

    log_rmse = np.sqrt(
        mean_squared_error(y_true_log, y_pred_log)
    )
    log_mae = mean_absolute_error(y_true_log, y_pred_log)

    raw_r2 = r2_score(y_true_price, y_pred_price)
    raw_mae = mean_absolute_error(y_true_price, y_pred_price)

    mape = np.mean(
        np.abs(y_true_price - y_pred_price)
        / np.maximum(np.abs(y_true_price), 1e-8)
    ) * 100

    print(f"\n{model_name}")
    print("-" * 55)
    print(f"Raw-price R2 : {raw_r2:.6f}")
    print(f"Log RMSE     : {log_rmse:.6f}")
    print(f"Log MAE      : {log_mae:.6f}")
    print(f"Raw-price MAE: Rs {raw_mae:,.2f}")
    print(f"MAPE         : {mape:.2f}%")

    return {
        "Model": model_name,
        "R2": raw_r2,
        "Log RMSE": log_rmse,
        "Log MAE": log_mae,
        "Raw MAE": raw_mae,
        "MAPE": mape
    }

# ============================================================
# 6. UNTUNED BASELINES
# ============================================================

xgb_baseline = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

et_baseline = ExtraTreesRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

print("\nTraining untuned baselines...")

xgb_baseline.fit(X_train, y_train)
et_baseline.fit(X_train, y_train)

results = []

results.append(
    evaluate(
        y_test,
        xgb_baseline.predict(X_test),
        "Untuned XGBoost"
    )
)

results.append(
    evaluate(
        y_test,
        et_baseline.predict(X_test),
        "Untuned Extra Trees"
    )
)

# ============================================================
# 7. XGBOOST HYPERPARAMETER TUNING
# ============================================================

xgb_param_dist = {
    "n_estimators": [400, 600, 800, 1000, 1200, 1500],
    "max_depth": randint(3, 12),
    "learning_rate": uniform(0.02, 0.13),
    "min_child_weight": randint(1, 9),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.5, 0.5),
    "gamma": uniform(0.0, 0.5),
    "reg_lambda": uniform(0.1, 5.0),
    "reg_alpha": uniform(0.0, 1.0)
}

xgb_search = RandomizedSearchCV(
    estimator=XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=1,
        verbosity=0
    ),
    param_distributions=xgb_param_dist,
    n_iter=30,
    scoring=kaggle_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    return_train_score=True,
    error_score="raise"
)

print("\nTuning XGBoost...")

xgb_search.fit(X_train, y_train)

print("\nBest XGBoost parameters:")
print(xgb_search.best_params_)

print(
    "Best CV raw-price R2:",
    round(xgb_search.best_score_, 6)
)

# Show the five best configurations.
xgb_cv_results = pd.DataFrame(xgb_search.cv_results_)

xgb_top5 = xgb_cv_results.sort_values(
    "rank_test_score"
)[[
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "rank_test_score",
    "params"
]].head(5)

print("\nTop 5 XGBoost configurations:")
print(xgb_top5.to_string(index=False))

results.append(
    evaluate(
        y_test,
        xgb_search.best_estimator_.predict(X_test),
        "Tuned XGBoost"
    )
)

# ============================================================
# 8. EXTRA TREES HYPERPARAMETER TUNING
# ============================================================

et_param_dist = {
    "n_estimators": [300, 500, 700, 1000],
    "max_depth": [None, 15, 20, 30, 40],
    "min_samples_leaf": [1, 2, 3, 4, 5],
    "min_samples_split": [2, 4, 6, 10],
    "max_features": [0.5, 0.7, 0.85, 1.0],
    "bootstrap": [True, False]
}

et_search = RandomizedSearchCV(
    estimator=ExtraTreesRegressor(
        random_state=42,
        n_jobs=1
    ),
    param_distributions=et_param_dist,
    n_iter=20,
    scoring=kaggle_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    return_train_score=True,
    error_score="raise"
)

print("\nTuning Extra Trees...")

et_search.fit(X_train, y_train)

print("\nBest Extra Trees parameters:")
print(et_search.best_params_)

print(
    "Best CV raw-price R2:",
    round(et_search.best_score_, 6)
)

et_cv_results = pd.DataFrame(et_search.cv_results_)

et_top5 = et_cv_results.sort_values(
    "rank_test_score"
)[[
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "rank_test_score",
    "params"
]].head(5)

print("\nTop 5 Extra Trees configurations:")
print(et_top5.to_string(index=False))

results.append(
    evaluate(
        y_test,
        et_search.best_estimator_.predict(X_test),
        "Tuned Extra Trees"
    )
)

# ============================================================
# 9. FINAL COMPARISON
# ============================================================

comparison = pd.DataFrame(results).sort_values(
    "R2",
    ascending=False
)

print("\nFINAL MODEL COMPARISON")
print(comparison.to_string(index=False))

print("\nBest model by held-out raw-price R2:")
print(comparison.iloc[0]["Model"])

# ============================================================
# 10. SAVE SEARCH RESULTS
# ============================================================

xgb_cv_results.to_csv("xgboost_tuning_results.csv", index=False)
et_cv_results.to_csv("extra_trees_tuning_results.csv", index=False)

print("\nSaved:")
print("- xgboost_tuning_results.csv")
print("- extra_trees_tuning_results.csv")


Dataset shape: (14431, 66)
Feature shape: (14431, 65)
Target range: 8.243282523048375 to 12.685958534005326

Training shape: (11544, 65)
Test shape: (2887, 65)

Training untuned baselines...

Untuned XGBoost
-------------------------------------------------------
Raw-price R2 : 0.916980
Log RMSE     : 0.145139
Log MAE      : 0.086203
Raw-price MAE: Rs 3,901.00
MAPE         : 8.65%

Untuned Extra Trees
-------------------------------------------------------
Raw-price R2 : 0.904125
Log RMSE     : 0.155412
Log MAE      : 0.084792
Raw-price MAE: Rs 3,906.08
MAPE         : 8.54%

Tuning XGBoost...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best XGBoost parameters:
{'colsample_bytree': np.float64(0.5852620618436457), 'gamma': np.float64(0.03252579649263976), 'learning_rate': np.float64(0.1433551198429333), 'max_depth': 4, 'min_child_weight': 2, 'n_estimators': 400, 'reg_alpha': np.float64(0.015966252220214194), 'reg_lambda': np.float64(1.254469128110745), 'subsample': np.

XGBoost has inconclusive/insignificant improvemnt 
but ExtraTree has improvement.